# TP — Manipuler une base SQLite avec Python et Pandas

**Auteur : A. Larhlimi**

SANAE ELRHARABY
BASMA EN-NIYARI

## Objectifs
À la fin de ce notebook, vous saurez :
- créer une base SQLite et un modèle relationnel ;
- insérer, consulter, filtrer, modifier et supprimer des données ;
- relier plusieurs tables avec `JOIN` ;
- utiliser `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, `GROUP BY` et `HAVING` ;
- exécuter et afficher les requêtes SQL avec `pandas`.

> Le notebook est autonome et conçu pour être exécuté de haut en bas dans Google Colab.

## 1. SQLite, `sqlite3` et Pandas

**SQLite** est un SGBD relationnel léger : toute la base peut être stockée dans un simple fichier. Contrairement à un SGBD client-serveur (PostgreSQL, MySQL…), il n'est pas nécessaire de lancer un serveur séparé.

- Une **connexion** ouvre l'accès à la base.
- Un **curseur** permet d'envoyer des instructions SQL.
- `commit()` valide les modifications de la transaction.
- `close()` ferme proprement la connexion.
- `pandas.read_sql_query()` permet d'exécuter un `SELECT` et de récupérer directement le résultat dans un `DataFrame`.

creation de la  base de **donnees**

In [1]:
import sqlite3
import pandas as pd # manipulation des tab
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DB_PATH = "gestion_ventes.db"

# Pour rendre le notebook reproductible, on supprime l'ancienne base
# si elle existe déjà, puis on la recrée.
if Path(DB_PATH).exists():
    Path(DB_PATH).unlink()

# Ouverture/création de la base SQLite.
conn = sqlite3.connect(DB_PATH)

# SQLite n'active pas les contraintes de clés étrangères par défaut
# pour chaque nouvelle connexion : on les active explicitement.
conn.execute("PRAGMA foreign_keys = ON;")

# Le curseur servira pour CREATE, INSERT, UPDATE, DELETE...
cursor = conn.cursor()

pd.DataFrame({
    "Élément": ["Base", "Connexion", "Clés étrangères"],
    "Valeur": [DB_PATH, "ouverte", "activées"]
})

,Élément,Valeur
0,Base,gestion_ventes.db
1,Connexion,ouverte
2,Clés étrangères,activées


## 2. Création du modèle relationnel

Nous utiliserons quatre tables :

- **clients** : informations sur les clients ;
- **produits** : catalogue des produits ;
- **commandes** : une commande appartient à un client ;
- **details_commandes** : lignes reliant commandes et produits.

Relations : `clients 1—N commandes`, `commandes 1—N details_commandes`, `produits 1—N details_commandes`.

Quelques contraintes importantes :
- `PRIMARY KEY` identifie chaque ligne ;
- `FOREIGN KEY` garantit les relations ;
- `NOT NULL` interdit une valeur absente ;
- `UNIQUE` interdit les doublons sur une colonne.

In [2]:
cursor.executescript("""
CREATE TABLE clients (
    id_client INTEGER PRIMARY KEY,
    nom TEXT NOT NULL,
    ville TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    telephone TEXT
);

CREATE TABLE produits (
    id_produit INTEGER PRIMARY KEY,
    nom_produit TEXT NOT NULL,
    categorie TEXT NOT NULL,
    prix REAL NOT NULL CHECK (prix >= 0),
    stock INTEGER NOT NULL CHECK (stock >= 0)
);

CREATE TABLE commandes (
    id_commande INTEGER PRIMARY KEY,
    id_client INTEGER NOT NULL,
    date_commande TEXT NOT NULL,
    statut TEXT NOT NULL,
    FOREIGN KEY (id_client) REFERENCES clients(id_client)
);

CREATE TABLE details_commandes (
    id_detail INTEGER PRIMARY KEY,
    id_commande INTEGER NOT NULL,
    id_produit INTEGER NOT NULL,
    quantite INTEGER NOT NULL CHECK (quantite > 0),
    prix_unitaire REAL NOT NULL CHECK (prix_unitaire >= 0),
    FOREIGN KEY (id_commande) REFERENCES commandes(id_commande),
    FOREIGN KEY (id_produit) REFERENCES produits(id_produit),
    UNIQUE (id_commande, id_produit)
);
""")

conn.commit()

# Affichage de la liste des tables avec Pandas.
pd.read_sql_query(
    "SELECT name AS table_sqlite FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)

,table_sqlite
0,clients
1,commandes
2,details_commandes
3,produits


**Interprétation :** les quatre tables du modèle commercial sont maintenant créées. Les clés étrangères protègent la cohérence entre clients, commandes, produits et lignes de commande.

## 3. Insertion de données

Nous allons combiner :
- `execute()` pour une insertion unique ;
- `executemany()` pour plusieurs lignes ;
- les paramètres `?`, qui séparent les données du texte SQL et évitent la concaténation dangereuse.

In [3]:
# Exemple avec execute() : insertion d'un premier client.
#code pour inserer une seue ligne
cursor.execute(
    "INSERT INTO clients (id_client, nom, ville, email, telephone) VALUES (?, ?, ?, ?, ?)",
    (1, "taha Benali", "Casablanca", "taha@example.com", None)
)

# Insertion de plusieurs clients avec executemany().
clients = [
    (2, "Youssef Alaoui", "Rabat", "youssef@example.com", "0610000002"),
    (3, "Sara Idrissi", "Settat", "sara@example.com", None),
    (4, "Omar Tazi", "Marrakech", "omar@example.com", "0610000004"),
    (5, "Lina El Fassi", "Casablanca", "lina@example.com", None),
    (6, "Mehdi Berrada", "Tanger", "mehdi@example.com", "0610000006"),
    (7, "Nora Amrani", "Settat", "nora@example.com", "0610000007"),
    (8, "Karim Chraibi", "Fès", "karim@example.com", None),

]
cursor.executemany(
    "INSERT INTO clients (id_client, nom, ville, email, telephone) VALUES (?, ?, ?, ?, ?)",
    clients
)

produits = [
    (1, "Ordinateur portable", "Informatique", 8500.00, 12),
    (2, "Souris sans fil", "Accessoires", 180.00, 60),
    (3, "Clavier mécanique", "Accessoires", 650.00, 35),
    (4, "Écran 27 pouces", "Informatique", 2400.00, 18),
    (5, "Casque audio", "Audio", 950.00, 30),
    (6, "Webcam HD", "Accessoires", 520.00, 25),
    (7, "SSD externe 1 To", "Stockage", 1100.00, 22),
    (8, "Microphone USB", "Audio", 1250.00, 15),
    (9, "Station d'accueil", "Accessoires", 1450.00, 10),
    (10, "Tablette graphique", "Création", 3200.00, 8),
    # Produit volontairement jamais commandé : utile pour LEFT JOIN.
    (11, "Projecteur portable", "Multimédia", 4800.00, 5)
]
cursor.executemany(
    "INSERT INTO produits VALUES (?, ?, ?, ?, ?)",
    produits
)

commandes = [
    (1, 1, "2026-01-12", "Livrée"),
    (2, 2, "2026-01-18", "Livrée"),
    (3, 1, "2026-02-03", "Livrée"),
    (4, 3, "2026-02-14", "Livrée"),
    (5, 4, "2026-03-02", "Livrée"),
    (6, 5, "2026-03-19", "Livrée"),
    (7, 2, "2026-04-05", "Livrée"),
    (8, 6, "2026-04-21", "En préparation"),
    (9, 1, "2026-05-09", "Livrée"),
    (10, 7, "2026-05-17", "Livrée"),
    (11, 3, "2026-06-08", "Annulée"),
    (12, 5, "2026-06-25", "Livrée")
]
cursor.executemany(
    "INSERT INTO commandes VALUES (?, ?, ?, ?)",
    commandes
)

details = [
    (1, 1, 1, 1, 8500.00), (2, 1, 2, 2, 180.00),
    (3, 2, 4, 1, 2400.00), (4, 2, 3, 1, 650.00),
    (5, 3, 5, 2, 950.00), (6, 3, 6, 1, 520.00),
    (7, 4, 7, 1, 1100.00), (8, 4, 2, 3, 180.00),
    (9, 5, 1, 1, 8500.00), (10, 5, 8, 1, 1250.00),
    (11, 6, 10, 1, 3200.00), (12, 6, 3, 2, 650.00),
    (13, 7, 9, 1, 1450.00), (14, 7, 2, 2, 180.00),
    (15, 8, 4, 2, 2400.00), (16, 8, 6, 2, 520.00),
    (17, 9, 7, 2, 1100.00), (18, 9, 8, 1, 1250.00),
    (19, 10, 5, 1, 950.00), (20, 10, 2, 1, 180.00),
    (21, 11, 1, 1, 8500.00), (22, 11, 3, 1, 650.00),
    (23, 12, 4, 1, 2400.00), (24, 12, 9, 1, 1450.00)
]
cursor.executemany(
    "INSERT INTO details_commandes VALUES (?, ?, ?, ?, ?)",
    details
)

conn.commit()
# affiche le nombre de ligne dans chaque entite
pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM clients) AS nb_clients,
    (SELECT COUNT(*) FROM produits) AS nb_produits,
    (SELECT COUNT(*) FROM commandes) AS nb_commandes,
    (SELECT COUNT(*) FROM details_commandes) AS nb_lignes
""", conn)

,nb_clients,nb_produits,nb_commandes,nb_lignes
0,8,11,12,24


**Interprétation :** le jeu de données contient plusieurs villes, catégories, dates, clients récurrents et un produit jamais commandé. Cela permettra de construire des requêtes réalistes.

## 4. Consultation avec `SELECT`

### 4.1 `SELECT *`
`*` demande toutes les colonnes de la table.

In [4]:
pd.read_sql_query("SELECT * FROM clients;", conn)

,id_client,nom,ville,email,telephone
0,1,taha Benali,Casablanca,taha@example.com,None
1,2,Youssef Alaoui,Rabat,youssef@example.com,0610000002
2,3,Sara Idrissi,Settat,sara@example.com,None
3,4,Omar Tazi,Marrakech,omar@example.com,0610000004
4,5,Lina El Fassi,Casablanca,lina@example.com,None
5,6,Mehdi Berrada,Tanger,mehdi@example.com,0610000006
6,7,Nora Amrani,Settat,nora@example.com,0610000007
7,8,Karim Chraibi,Fès,karim@example.com,None


### 4.2 Sélectionner seulement certaines colonnes
On peut limiter le résultat aux informations utiles.

In [5]:
pd.read_sql_query(
    "SELECT nom_produit, categorie, prix as T FROM produits;",
    conn
)

,nom_produit,categorie,T
0,Ordinateur portable,Informatique,"8,500.00"
1,Souris sans fil,Accessoires,180.00
2,Clavier mécanique,Accessoires,650.00
3,Écran 27 pouces,Informatique,"2,400.00"
4,Casque audio,Audio,950.00
5,Webcam HD,Accessoires,520.00
6,SSD externe 1 To,Stockage,"1,100.00"
7,Microphone USB,Audio,"1,250.00"
8,Station d'accueil,Accessoires,"1,450.00"
9,Tablette graphique,Création,"3,200.00"


### 4.3 `DISTINCT`
`DISTINCT` élimine les valeurs dupliquées dans le résultat.

In [6]:
pd.read_sql_query(
    "SELECT DISTINCT categorie FROM produits ORDER BY categorie;",
    conn
)

,categorie
0,Accessoires
1,Audio
2,Création
3,Informatique
4,Multimédia
5,Stockage


### 4.4 `WHERE`, `ORDER BY` et `LIMIT`
Cherchons les trois produits les plus chers dont le prix dépasse 1 000 DH.

In [7]:
requete = """
SELECT nom_produit, categorie, prix
FROM produits
WHERE prix > ?
ORDER BY prix DESC
LIMIT 3;
"""

pd.read_sql_query(requete, conn, params=(1000,))

,nom_produit,categorie,prix
0,Ordinateur portable,Informatique,"8,500.00"
1,Projecteur portable,Multimédia,"4,800.00"
2,Tablette graphique,Création,"3,200.00"


**Interprétation :** `WHERE` filtre les lignes avant l'affichage, `ORDER BY ... DESC` classe du plus grand au plus petit et `LIMIT 3` ne conserve que trois résultats.

## 5. Filtrage des données

Les opérateurs suivants permettent de construire des conditions plus précises.

In [8]:
# AND : produits informatiques coûtant plus de 2 000 DH.
pd.read_sql_query("""
SELECT nom_produit, categorie, prix
FROM produits
WHERE categorie = 'Informatique' AND prix > 2000
ORDER BY prix DESC;
""", conn)

,nom_produit,categorie,prix
0,Ordinateur portable,Informatique,"8,500.00"
1,Écran 27 pouces,Informatique,"2,400.00"


In [10]:
# OR et NOT : clients situés à Casablanca ou Settat,
# sauf un client précis.
pd.read_sql_query("""
SELECT nom, ville
FROM clients
WHERE (ville = 'Casablanca' OR ville = 'Settat')
  AND NOT nom = 'Nora Amrani'
ORDER BY ville, nom;
""", conn)

,nom,ville
0,Lina El Fassi,Casablanca
1,taha Benali,Casablanca
2,Sara Idrissi,Settat


In [11]:
# IN : plusieurs catégories autorisées.
pd.read_sql_query("""
SELECT nom_produit, categorie, prix
FROM produits
WHERE categorie IN ('Audio', 'Stockage', 'Création')
ORDER BY categorie, prix;
""", conn)

,nom_produit,categorie,prix
0,Casque audio,Audio,950.00
1,Microphone USB,Audio,"1,250.00"
2,Tablette graphique,Création,"3,200.00"
3,SSD externe 1 To,Stockage,"1,100.00"


In [12]:
# BETWEEN : intervalle inclusif.
pd.read_sql_query("""
SELECT nom_produit, prix
FROM produits
WHERE prix BETWEEN 500 AND 1500
ORDER BY prix;
""", conn)

,nom_produit,prix
0,Webcam HD,520.00
1,Clavier mécanique,650.00
2,Casque audio,950.00
3,SSD externe 1 To,"1,100.00"
4,Microphone USB,"1,250.00"
5,Station d'accueil,"1,450.00"


In [13]:
# LIKE : recherche textuelle.
pd.read_sql_query("""
SELECT nom_produit, categorie
FROM produits
WHERE nom_produit LIKE '%USB%' OR nom_produit LIKE '%sans fil%';
""", conn)

,nom_produit,categorie
0,Souris sans fil,Accessoires
1,Microphone USB,Audio


In [14]:
# IS NULL : clients dont le téléphone n'est pas renseigné.
pd.read_sql_query("""
SELECT nom, ville, telephone
FROM clients
WHERE telephone IS NULL;
""", conn)

,nom,ville,telephone
0,taha Benali,Casablanca,None
1,Sara Idrissi,Settat,None
2,Lina El Fassi,Casablanca,None
3,Karim Chraibi,Fès,None


**Interprétation :** `AND` exige plusieurs conditions simultanément, `OR` accepte l'une ou l'autre, `IN` teste une liste, `BETWEEN` un intervalle, `LIKE` un motif textuel et `IS NULL` l'absence de valeur.

## 6. Modification avec `UPDATE`

Pour bien voir l'effet de la modification, affichons d'abord le produit concerné, modifions son stock avec une requête paramétrée, puis réaffichons-le.

In [15]:
# Avant modification
pd.read_sql_query(
    "SELECT id_produit, nom_produit, stock FROM produits WHERE id_produit = 6;",
    conn
)

,id_produit,nom_produit,stock
0,6,Webcam HD,25


In [16]:
# Mise à jour paramétrée.
nouveau_stock = 40
id_produit = 6

cursor.execute(
    "UPDATE produits SET stock = ? WHERE id_produit = ?",
    (nouveau_stock, id_produit)
)
conn.commit()

# Après modification
pd.read_sql_query(
    "SELECT id_produit, nom_produit, stock FROM produits WHERE id_produit = 6;",
    conn
)

,id_produit,nom_produit,stock
0,6,Webcam HD,40


**Interprétation :** seule la ligne dont `id_produit = 6` a été modifiée. Une clause `WHERE` oubliée dans un `UPDATE` modifierait toutes les lignes.

## 7. Suppression avec `DELETE`

Pour ne pas perturber les analyses suivantes, nous créons un client temporaire sans commande, puis nous le supprimons.

In [17]:
# Ajout d'un client temporaire.
cursor.execute(
    "INSERT INTO clients (id_client, nom, ville, email, telephone) VALUES (?, ?, ?, ?, ?)",
    (99, "Client Temporaire", "Settat", "temporaire@example.com", None)
)
conn.commit()

pd.read_sql_query("SELECT * FROM clients WHERE id_client = 99;", conn)

,id_client,nom,ville,email,telephone
0,99,Client Temporaire,Settat,temporaire@example.com,None


In [18]:
# Suppression ciblée.
cursor.execute("DELETE FROM clients WHERE id_client = ?", (99,))
conn.commit()

# Le DataFrame vide confirme la suppression.
pd.read_sql_query("SELECT * FROM clients WHERE id_client = 99;", conn)

,id_client,nom,ville,email,telephone


## 8. Jointures

Une jointure combine des lignes provenant de plusieurs tables grâce à leurs clés.

### 8.1 `INNER JOIN` — clients ayant passé des commandes
La jointure relie `clients.id_client` à `commandes.id_client`.

In [19]:
pd.read_sql_query("""
SELECT c.nom AS client, co.id_commande, co.date_commande, co.statut
FROM clients AS c
INNER JOIN commandes AS co
    ON c.id_client = co.id_client
ORDER BY co.date_commande;
""", conn)

,client,id_commande,date_commande,statut
0,taha Benali,1,2026-01-12,Livrée
1,Youssef Alaoui,2,2026-01-18,Livrée
2,taha Benali,3,2026-02-03,Livrée
3,Sara Idrissi,4,2026-02-14,Livrée
4,Omar Tazi,5,2026-03-02,Livrée
5,Lina El Fassi,6,2026-03-19,Livrée
6,Youssef Alaoui,7,2026-04-05,Livrée
7,Mehdi Berrada,8,2026-04-21,En préparation
8,taha Benali,9,2026-05-09,Livrée
9,Nora Amrani,10,2026-05-17,Livrée


### 8.2 Produits commandés et montant des lignes

Une ligne de commande référence un produit par `id_produit`. Le montant de ligne est calculé par `quantite × prix_unitaire`.

In [20]:
pd.read_sql_query("""
SELECT
    dc.id_commande,
    p.nom_produit,
    dc.quantite,
    dc.prix_unitaire,
    ROUND(dc.quantite * dc.prix_unitaire, 2) AS montant_ligne
FROM details_commandes AS dc
INNER JOIN produits AS p
    ON dc.id_produit = p.id_produit
ORDER BY dc.id_commande, p.nom_produit;
""", conn)

,id_commande,nom_produit,quantite,prix_unitaire,montant_ligne
0,1,Ordinateur portable,1,"8,500.00","8,500.00"
1,1,Souris sans fil,2,180.00,360.00
2,2,Clavier mécanique,1,650.00,650.00
3,2,Écran 27 pouces,1,"2,400.00","2,400.00"
4,3,Casque audio,2,950.00,"1,900.00"
5,3,Webcam HD,1,520.00,520.00
6,4,SSD externe 1 To,1,"1,100.00","1,100.00"
7,4,Souris sans fil,3,180.00,540.00
8,5,Microphone USB,1,"1,250.00","1,250.00"
9,5,Ordinateur portable,1,"8,500.00","8,500.00"


### 8.3 `LEFT JOIN` — conserver aussi les produits jamais commandés

Contrairement à `INNER JOIN`, `LEFT JOIN` conserve toutes les lignes de la table située à gauche.

In [21]:
pd.read_sql_query("""
SELECT
    p.nom_produit,
    COALESCE(SUM(dc.quantite), 0) AS quantite_commandee
FROM produits AS p
LEFT JOIN details_commandes AS dc
    ON p.id_produit = dc.id_produit
LEFT JOIN commandes AS co
    ON dc.id_commande = co.id_commande
   AND co.statut <> 'Annulée'
GROUP BY p.id_produit, p.nom_produit
ORDER BY quantite_commandee DESC, p.nom_produit;
""", conn)

,nom_produit,quantite_commandee
0,Souris sans fil,8
1,Clavier mécanique,4
2,Écran 27 pouces,4
3,Casque audio,3
4,Ordinateur portable,3
5,SSD externe 1 To,3
6,Webcam HD,3
7,Microphone USB,2
8,Station d'accueil,2
9,Tablette graphique,1


**Interprétation :** le produit jamais commandé reste visible avec une quantité égale à 0 grâce à `LEFT JOIN` et `COALESCE`.

## 9. Fonctions d'agrégation

Les fonctions d'agrégation résument plusieurs lignes :
- `COUNT()` compte ;
- `SUM()` additionne ;
- `AVG()` calcule une moyenne ;
- `MIN()` et `MAX()` donnent les extrêmes.

Dans les analyses de ventes suivantes, les commandes **annulées sont exclues du chiffre d'affaires**.

In [22]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS nombre_clients
FROM clients;
""", conn)

,nombre_clients
0,8


In [23]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS nombre_commandes,
    COUNT(DISTINCT id_client) AS clients_ayant_commande
FROM commandes
WHERE statut <> 'Annulée';
""", conn)

,nombre_commandes,clients_ayant_commande
0,11,7


In [24]:
pd.read_sql_query("""
SELECT
    ROUND(AVG(prix), 2) AS prix_moyen,
    MIN(prix) AS prix_minimum,
    MAX(prix) AS prix_maximum
FROM produits;
""", conn)

,prix_moyen,prix_minimum,prix_maximum
0,"2,272.73",180.00,"8,500.00"


In [25]:
# Chiffre d'affaires total : somme des montants des lignes
# appartenant aux commandes non annulées.
pd.read_sql_query("""
SELECT
    ROUND(SUM(dc.quantite * dc.prix_unitaire), 2) AS chiffre_affaires_total
FROM details_commandes AS dc
JOIN commandes AS co ON co.id_commande = dc.id_commande
WHERE co.statut <> 'Annulée';
""", conn)

,chiffre_affaires_total
0,"46,300.00"


In [26]:
# Le panier moyen doit d'abord calculer le montant de chaque commande,
# puis faire la moyenne de ces montants.
pd.read_sql_query("""
WITH montants_commandes AS (
    SELECT
        co.id_commande,
        SUM(dc.quantite * dc.prix_unitaire) AS montant
    FROM commandes AS co
    JOIN details_commandes AS dc
        ON co.id_commande = dc.id_commande
    WHERE co.statut <> 'Annulée'
    GROUP BY co.id_commande
)
SELECT ROUND(AVG(montant), 2) AS panier_moyen
FROM montants_commandes;
""", conn)

,panier_moyen
0,"4,209.09"


## 10. `GROUP BY` et `HAVING`

`GROUP BY` forme des groupes de lignes afin de calculer un agrégat pour chaque groupe.

**Différence fondamentale :**
- `WHERE` filtre les **lignes avant** le regroupement ;
- `HAVING` filtre les **groupes après** `GROUP BY`.

### 10.1 Chiffre d'affaires par client

In [27]:
pd.read_sql_query("""
SELECT
    c.id_client,
    c.nom,
    ROUND(SUM(dc.quantite * dc.prix_unitaire), 2) AS chiffre_affaires
FROM clients AS c
JOIN commandes AS co ON c.id_client = co.id_client
JOIN details_commandes AS dc ON co.id_commande = dc.id_commande
WHERE co.statut <> 'Annulée'
GROUP BY c.id_client, c.nom
ORDER BY chiffre_affaires DESC;
""", conn)

,id_client,nom,chiffre_affaires
0,1,taha Benali,"14,730.00"
1,4,Omar Tazi,"9,750.00"
2,5,Lina El Fassi,"8,350.00"
3,6,Mehdi Berrada,"5,840.00"
4,2,Youssef Alaoui,"4,860.00"
5,3,Sara Idrissi,"1,640.00"
6,7,Nora Amrani,"1,130.00"


### 10.2 Chiffre d'affaires et quantité vendue par produit

In [28]:
pd.read_sql_query("""
SELECT
    p.nom_produit,
    COALESCE(SUM(CASE WHEN co.statut <> 'Annulée' THEN dc.quantite ELSE 0 END), 0) AS quantite_vendue,
    ROUND(COALESCE(SUM(CASE WHEN co.statut <> 'Annulée'
                            THEN dc.quantite * dc.prix_unitaire ELSE 0 END), 0), 2) AS chiffre_affaires
FROM produits AS p
LEFT JOIN details_commandes AS dc ON p.id_produit = dc.id_produit
LEFT JOIN commandes AS co ON dc.id_commande = co.id_commande
GROUP BY p.id_produit, p.nom_produit
ORDER BY chiffre_affaires DESC;
""", conn)

,nom_produit,quantite_vendue,chiffre_affaires
0,Ordinateur portable,2,"17,000.00"
1,Écran 27 pouces,4,"9,600.00"
2,SSD externe 1 To,3,"3,300.00"
3,Tablette graphique,1,"3,200.00"
4,Station d'accueil,2,"2,900.00"
5,Casque audio,3,"2,850.00"
6,Microphone USB,2,"2,500.00"
7,Clavier mécanique,3,"1,950.00"
8,Webcam HD,3,"1,560.00"
9,Souris sans fil,8,"1,440.00"


### 10.3 Chiffre d'affaires par catégorie

In [29]:
pd.read_sql_query("""
SELECT
    p.categorie,
    ROUND(SUM(dc.quantite * dc.prix_unitaire), 2) AS chiffre_affaires
FROM produits AS p
JOIN details_commandes AS dc ON p.id_produit = dc.id_produit
JOIN commandes AS co ON dc.id_commande = co.id_commande
WHERE co.statut <> 'Annulée'
GROUP BY p.categorie
ORDER BY chiffre_affaires DESC;
""", conn)

,categorie,chiffre_affaires
0,Informatique,"26,600.00"
1,Accessoires,"7,850.00"
2,Audio,"5,350.00"
3,Stockage,"3,300.00"
4,Création,"3,200.00"


### 10.4 Nombre de commandes par client

In [30]:
pd.read_sql_query("""
SELECT
    c.nom,
    COUNT(co.id_commande) AS nombre_commandes
FROM clients AS c
LEFT JOIN commandes AS co
    ON c.id_client = co.id_client
   AND co.statut <> 'Annulée'
GROUP BY c.id_client, c.nom
ORDER BY nombre_commandes DESC, c.nom;
""", conn)

,nom,nombre_commandes
0,taha Benali,3
1,Lina El Fassi,2
2,Youssef Alaoui,2
3,Mehdi Berrada,1
4,Nora Amrani,1
5,Omar Tazi,1
6,Sara Idrissi,1
7,Karim Chraibi,0


### 10.5 Panier moyen par client

In [31]:
pd.read_sql_query("""
WITH montant_par_commande AS (
    SELECT
        co.id_commande,
        co.id_client,
        SUM(dc.quantite * dc.prix_unitaire) AS montant
    FROM commandes AS co
    JOIN details_commandes AS dc ON co.id_commande = dc.id_commande
    WHERE co.statut <> 'Annulée'
    GROUP BY co.id_commande, co.id_client
)
SELECT
    c.nom,
    ROUND(AVG(mpc.montant), 2) AS panier_moyen
FROM clients AS c
JOIN montant_par_commande AS mpc ON c.id_client = mpc.id_client
GROUP BY c.id_client, c.nom
ORDER BY panier_moyen DESC;
""", conn)

,nom,panier_moyen
0,Omar Tazi,"9,750.00"
1,Mehdi Berrada,"5,840.00"
2,taha Benali,"4,910.00"
3,Lina El Fassi,"4,175.00"
4,Youssef Alaoui,"2,430.00"
5,Sara Idrissi,"1,640.00"
6,Nora Amrani,"1,130.00"


### 10.6 `HAVING` — clients dont le CA dépasse 5 000 DH

In [32]:
pd.read_sql_query("""
SELECT
    c.nom,
    ROUND(SUM(dc.quantite * dc.prix_unitaire), 2) AS chiffre_affaires
FROM clients AS c
JOIN commandes AS co ON c.id_client = co.id_client
JOIN details_commandes AS dc ON co.id_commande = dc.id_commande
WHERE co.statut <> 'Annulée'
GROUP BY c.id_client, c.nom
HAVING SUM(dc.quantite * dc.prix_unitaire) > 5000
ORDER BY chiffre_affaires DESC;
""", conn)

,nom,chiffre_affaires
0,taha Benali,"14,730.00"
1,Omar Tazi,"9,750.00"
2,Lina El Fassi,"8,350.00"
3,Mehdi Berrada,"5,840.00"


**Interprétation :** `WHERE` a d'abord exclu les commandes annulées ; `HAVING` a ensuite conservé uniquement les groupes de clients dont le chiffre d'affaires calculé dépasse 5 000 DH.

## 11. Requête finale de synthèse

Construisons un tableau de pilotage client combinant :
- plusieurs `JOIN` ;
- un filtre `WHERE` ;
- `COUNT`, `SUM` et calculs ;
- `GROUP BY` ;
- `HAVING` ;
- `ORDER BY`.

Le **panier moyen** est calculé comme `CA / nombre de commandes distinctes`.

In [33]:
requete_synthese = """
SELECT
    c.nom AS client,
    COUNT(DISTINCT co.id_commande) AS nombre_commandes,
    SUM(dc.quantite) AS quantite_totale_achetee,
    ROUND(SUM(dc.quantite * dc.prix_unitaire), 2) AS chiffre_affaires,
    ROUND(
        SUM(dc.quantite * dc.prix_unitaire)
        / COUNT(DISTINCT co.id_commande),
        2
    ) AS panier_moyen
FROM clients AS c
JOIN commandes AS co
    ON c.id_client = co.id_client
JOIN details_commandes AS dc
    ON co.id_commande = dc.id_commande
WHERE co.statut <> 'Annulée'
GROUP BY c.id_client, c.nom
HAVING SUM(dc.quantite * dc.prix_unitaire) >= 1000
ORDER BY chiffre_affaires DESC, nombre_commandes DESC;
"""

pd.read_sql_query(requete_synthese, conn)

,client,nombre_commandes,quantite_totale_achetee,chiffre_affaires,panier_moyen
0,taha Benali,3,9,"14,730.00","4,910.00"
1,Omar Tazi,1,2,"9,750.00","9,750.00"
2,Lina El Fassi,2,5,"8,350.00","4,175.00"
3,Mehdi Berrada,1,4,"5,840.00","5,840.00"
4,Youssef Alaoui,2,5,"4,860.00","2,430.00"
5,Sara Idrissi,1,4,"1,640.00","1,640.00"
6,Nora Amrani,1,2,"1,130.00","1,130.00"


**Interprétation :** ce tableau synthétise l'activité commerciale par client. Il illustre comment SQL peut transformer des données transactionnelles réparties sur plusieurs tables en indicateurs métier directement exploitables.

## 12. Bonnes pratiques

### Requêtes paramétrées
Préférer :
`cursor.execute("... WHERE id = ?", (valeur,))`

à la concaténation de chaînes. Les paramètres réduisent notamment le risque d'**injection SQL** et gèrent correctement l'échappement des valeurs.

### Transactions
Plusieurs modifications liées peuvent être regroupées dans une transaction. `commit()` valide ; `rollback()` annule en cas d'erreur.

### Intégrité
Les clés primaires identifient les lignes. Les clés étrangères empêchent la création de relations incohérentes lorsqu'elles sont activées dans SQLite.

### Fermeture
Une connexion doit être fermée lorsqu'elle n'est plus nécessaire. Nous le ferons à la toute fin, après les exercices et leurs corrigés.

In [34]:
# Exemple pédagogique de transaction avec rollback :
# la modification est volontairement annulée.
stock_avant = pd.read_sql_query(
    "SELECT stock FROM produits WHERE id_produit = 2;", conn
).iloc[0, 0]

try:
    cursor.execute("UPDATE produits SET stock = stock - ? WHERE id_produit = ?", (2, 2))
    # Ici, d'autres opérations pourraient suivre.
    conn.rollback()  # Démonstration : on annule volontairement.
except Exception:
    conn.rollback()
    raise

stock_apres = pd.read_sql_query(
    "SELECT stock FROM produits WHERE id_produit = 2;", conn
).iloc[0, 0]

pd.DataFrame({
    "Moment": ["Avant la transaction", "Après rollback"],
    "Stock": [stock_avant, stock_apres]
})

,Moment,Stock
0,Avant la transaction,60
1,Après rollback,60


## 13. Exercices

Essayez de résoudre les exercices suivants **avant d'ouvrir la section Corrigé**.

1. Afficher les trois produits les plus chers.
2. Calculer le nombre de produits par catégorie.
3. Trouver le client ayant effectué le plus de commandes non annulées.
4. Calculer le chiffre d'affaires par mois.
5. Trouver les produits jamais commandés.
6. Trouver les clients dont le chiffre d'affaires est supérieur au chiffre d'affaires moyen par client.
7. Calculer, pour chaque ville, le nombre de clients et le chiffre d'affaires réalisé.
8. Afficher les catégories dont le chiffre d'affaires dépasse 3 000 DH.

# Corrigé des exercices

### Exercice 1 — Trois produits les plus chers

In [35]:
pd.read_sql_query("""
SELECT nom_produit, prix
FROM produits
ORDER BY prix DESC
LIMIT 3;
""", conn)

,nom_produit,prix
0,Ordinateur portable,"8,500.00"
1,Projecteur portable,"4,800.00"
2,Tablette graphique,"3,200.00"


### Exercice 2 — Nombre de produits par catégorie

In [36]:
pd.read_sql_query("""
SELECT categorie, COUNT(*) AS nombre_produits
FROM produits
GROUP BY categorie
ORDER BY nombre_produits DESC, categorie;
""", conn)

,categorie,nombre_produits
0,Accessoires,4
1,Audio,2
2,Informatique,2
3,Création,1
4,Multimédia,1
5,Stockage,1


### Exercice 3 — Client ayant effectué le plus de commandes

In [37]:
pd.read_sql_query("""
SELECT c.nom, COUNT(co.id_commande) AS nombre_commandes
FROM clients AS c
JOIN commandes AS co ON c.id_client = co.id_client
WHERE co.statut <> 'Annulée'
GROUP BY c.id_client, c.nom
ORDER BY nombre_commandes DESC, c.nom
LIMIT 1;
""", conn)

,nom,nombre_commandes
0,taha Benali,3


### Exercice 4 — Chiffre d'affaires par mois

In [38]:
pd.read_sql_query("""
SELECT
    strftime('%Y-%m', co.date_commande) AS mois,
    ROUND(SUM(dc.quantite * dc.prix_unitaire), 2) AS chiffre_affaires
FROM commandes AS co
JOIN details_commandes AS dc ON co.id_commande = dc.id_commande
WHERE co.statut <> 'Annulée'
GROUP BY strftime('%Y-%m', co.date_commande)
ORDER BY mois;
""", conn)

,mois,chiffre_affaires
0,2026-01,"11,910.00"
1,2026-02,"4,060.00"
2,2026-03,"14,250.00"
3,2026-04,"7,650.00"
4,2026-05,"4,580.00"
5,2026-06,"3,850.00"


### Exercice 5 — Produits jamais commandés

In [39]:
pd.read_sql_query("""
SELECT p.id_produit, p.nom_produit
FROM produits AS p
LEFT JOIN details_commandes AS dc ON p.id_produit = dc.id_produit
WHERE dc.id_detail IS NULL;
""", conn)

,id_produit,nom_produit
0,11,Projecteur portable


### Exercice 6 — Clients au-dessus du CA moyen par client

In [40]:
pd.read_sql_query("""
WITH ca_clients AS (
    SELECT
        c.id_client,
        c.nom,
        SUM(dc.quantite * dc.prix_unitaire) AS ca
    FROM clients AS c
    JOIN commandes AS co ON c.id_client = co.id_client
    JOIN details_commandes AS dc ON co.id_commande = dc.id_commande
    WHERE co.statut <> 'Annulée'
    GROUP BY c.id_client, c.nom
)
SELECT nom, ROUND(ca, 2) AS chiffre_affaires
FROM ca_clients
WHERE ca > (SELECT AVG(ca) FROM ca_clients)
ORDER BY ca DESC;
""", conn)

,nom,chiffre_affaires
0,taha Benali,"14,730.00"
1,Omar Tazi,"9,750.00"
2,Lina El Fassi,"8,350.00"


### Exercice 7 — Clients et chiffre d'affaires par ville

In [41]:
pd.read_sql_query("""
SELECT
    c.ville,
    COUNT(DISTINCT c.id_client) AS nombre_clients,
    ROUND(COALESCE(SUM(
        CASE WHEN co.statut <> 'Annulée'
             THEN dc.quantite * dc.prix_unitaire
             ELSE 0 END
    ), 0), 2) AS chiffre_affaires
FROM clients AS c
LEFT JOIN commandes AS co ON c.id_client = co.id_client
LEFT JOIN details_commandes AS dc ON co.id_commande = dc.id_commande
GROUP BY c.ville
ORDER BY chiffre_affaires DESC;
""", conn)

,ville,nombre_clients,chiffre_affaires
0,Casablanca,2,"23,080.00"
1,Marrakech,1,"9,750.00"
2,Tanger,1,"5,840.00"
3,Rabat,1,"4,860.00"
4,Settat,2,"2,770.00"
5,Fès,1,0.00


### Exercice 8 — Catégories dont le CA dépasse 3 000 DH

In [42]:
pd.read_sql_query("""
SELECT
    p.categorie,
    ROUND(SUM(dc.quantite * dc.prix_unitaire), 2) AS chiffre_affaires
FROM produits AS p
JOIN details_commandes AS dc ON p.id_produit = dc.id_produit
JOIN commandes AS co ON dc.id_commande = co.id_commande
WHERE co.statut <> 'Annulée'
GROUP BY p.categorie
HAVING SUM(dc.quantite * dc.prix_unitaire) > 3000
ORDER BY chiffre_affaires DESC;
""", conn)

,categorie,chiffre_affaires
0,Informatique,"26,600.00"
1,Accessoires,"7,850.00"
2,Audio,"5,350.00"
3,Stockage,"3,300.00"
4,Création,"3,200.00"


## 14. Conclusion

Vous avez parcouru une chaîne complète :

**création → insertion → SELECT → filtrage → UPDATE/DELETE → JOIN → agrégations → GROUP BY/HAVING → analyse métier**

SQLite est particulièrement adapté à l'apprentissage, au prototypage et aux applications locales. Les mêmes concepts SQL constituent ensuite une base solide pour travailler avec PostgreSQL, MySQL ou d'autres SGBD relationnels.

In [43]:
# Fermeture propre de la connexion.
conn.close()

pd.DataFrame({
    "Ressource": ["Connexion SQLite"],
    "État final": ["fermée proprement"]
})

,Ressource,État final
0,Connexion SQLite,fermée proprement
